# Density estimation: parametric and kernel density estimation
Estimating a PDF from data, first by fitting a normal distribution, then with a KDE.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
import seaborn.objects as so
from scipy import stats
from sklearn.neighbors import KernelDensity

# fixed seed: the same data on every run
rng = np.random.default_rng(42)

## 1. Parametric density estimation
We draw 1000 values from a normal distribution with mean 50 and standard deviation 5, then pretend we do not know that.

In [ ]:
sample = rng.normal(loc=50, scale=5, size=1000)
# the sample mean and std are close to 50 and 5, not equal
print(sample.mean(), sample.std())

In [ ]:
# step 1: look at the shape. histnorm puts the bars on the density scale (total area 1)
go.Figure(go.Histogram(x=sample, nbinsx=10, histnorm='probability density'))

In [ ]:
# step 2: estimate the parameters from the sample
sample_mean = sample.mean()
sample_std = sample.std()
# step 3: plug them into the normal PDF
dist = stats.norm(sample_mean, sample_std)

# evaluate the fitted PDF at 100 points from the smallest to the largest value
values = np.linspace(sample.min(), sample.max(), 100)
densities = dist.pdf(values)        # one density per point, no loop needed
print(sample.min(), sample.max())
print('density at 50:', dist.pdf(50), ' true:', stats.norm(50, 5).pdf(50))

In [ ]:
fig = go.Figure()
fig.add_histogram(x=sample, nbinsx=10, histnorm='probability density', name='data')
fig.add_scatter(x=values, y=densities, name='fitted normal')
# badly chosen parameters give a curve that misses the data
fig.add_scatter(x=values, y=stats.norm(60, 12).pdf(values), name='mean 60, std 12')
fig

In [ ]:
# Figure 3: the first 30 and 100 values give s = 3.8, far below 5. Is that usual?
# 10,000 fresh samples of each size: average error of s, and how often s <= 3.86
r = np.random.default_rng(0)
S = r.normal(50, 5, size=(10_000, 1000))
for n in [10, 30, 100, 1000]:
    s = S[:, :n].std(axis=1)
    print(n, 'mean |s - 5| =', round(np.mean(np.abs(s - 5)), 2),
          ' share with s <= 3.86 =', round(np.mean(s <= 3.86), 4))

10 mean |s - 5| = 0.94  share with s <= 3.86 = 0.2512
30 mean |s - 5| = 0.53  share with s <= 3.86 = 0.0519
100 mean |s - 5| = 0.28  share with s <= 3.86 = 0.0008
1000 mean |s - 5| = 0.09  share with s <= 3.86 = 0.0


## 2. Data no famous distribution fits
300 values around 20 and 700 around 40: two peaks.

In [ ]:
data = np.concatenate([rng.normal(20, 5, 300), rng.normal(40, 5, 700)])
print(data.mean(), data.std(), data.min(), data.max())
go.Figure(go.Histogram(x=data, nbinsx=30, histnorm='probability density'))

## 3. KDE by hand on six points

In [ ]:
pts = np.array([2, 2.5, 3, 4, 8, 8.5])
h = 1.0                         # bandwidth: the std of each Gaussian bump
x = 3
# height of each point's bump at x
bumps = stats.norm.pdf((x - pts) / h)
print(bumps.round(4), bumps.sum().round(4))
# divide by n * h so that the total area is 1
print('KDE at 3:', bumps.sum() / (len(pts) * h))

In [ ]:
# the same with scikit-learn: it expects a 2D array (one row per point)
kd = KernelDensity(kernel='gaussian', bandwidth=1).fit(pts.reshape(-1, 1))
# score_samples returns the LOG of the density; exp undoes the log
print(np.exp(kd.score_samples(np.array([[3.0], [6.0], [8.0]]))))

## 4. KDE with scikit-learn on the two-peaked data

In [ ]:
grid = np.linspace(data.min(), data.max(), 100).reshape(-1, 1)
fig = go.Figure()
fig.add_histogram(x=data, nbinsx=30, histnorm='probability density', name='data', opacity=0.4)
for bw in [0.5, 3, 5]:
    model = KernelDensity(kernel='gaussian', bandwidth=bw)
    model.fit(data.reshape(-1, 1))
    dens = np.exp(model.score_samples(grid))
    fig.add_scatter(x=grid[:, 0], y=dens, name=f'bandwidth {bw}')
fig

In [ ]:
# density in the valley (30) and at the peaks: a large bandwidth fills the valley
for bw in [0.5, 3, 5]:
    m = KernelDensity(bandwidth=bw).fit(data.reshape(-1, 1))
    print(bw, np.exp(m.score_samples(np.array([[20.0], [30.0], [40.0]]))).round(4))

## 5. KDE with seaborn
`bw_adjust` multiplies seaborn's default bandwidth (Scott's rule); it is not a bandwidth itself.

In [ ]:
# Scott's rule bandwidth, the one bw_adjust = 1 uses
print('default bandwidth:', stats.gaussian_kde(data).factor * data.std(ddof=1))
df = pd.DataFrame({'value': data})
(so.Plot(df, x='value')
   .add(so.Line(), so.KDE(bw_adjust=0.2), label='bw_adjust 0.2')
   .add(so.Line(), so.KDE(bw_adjust=1), label='bw_adjust 1')
   .add(so.Line(), so.KDE(bw_adjust=2), label='bw_adjust 2'))

In [ ]:
# why log densities: far from the data the density underflows to 0, the log does not
kd6 = KernelDensity(bandwidth=1).fit(pts.reshape(-1, 1))
logs = kd6.score_samples(np.array([[3.0], [60.0]]))
print('log density:', logs.round(1), ' exp:', np.exp(logs))
print('smallest positive float64:', np.finfo(float).tiny)

In [ ]:
# more data at a FIXED thin bandwidth (0.5): count local peaks of the KDE
g = np.linspace(0, 60, 1201)
for n in [100, 1000, 10_000, 100_000]:
    counts = []
    for seed in range(10):
        r = np.random.default_rng(seed)
        d = np.concatenate([r.normal(20, 5, int(0.3 * n)), r.normal(40, 5, n - int(0.3 * n))])
        f = stats.gaussian_kde(d, bw_method=0.5 / d.std(ddof=1))(g)   # bandwidth = 0.5
        counts.append(((f[1:-1] > f[:-2]) & (f[1:-1] > f[2:])).sum())
    print(n, 'points: mean number of peaks =', np.mean(counts))